In [4]:
import collections
if not hasattr(collections, "Mapping"):
        import collections.abc
        collections.Mapping = collections.abc.Mapping
from experta import *

ERROR! Session/line number was not unique in database. History logging moved to new session 173


In [5]:
class ModelArchitecture(Fact):
    Architecture = Field(str, mandatory = True)
    num_layers = Field(int, mandatory=True)
    parameter_count = Field(int, mandatory=True)
    uses_batch_norm = Field(bool, default=False)
    uses_attention = Field(bool, default=False)

class Dataset(Fact):
    data_type = Field(str, mandatory=True)
    sample_count = Field(int, mandatory=True)
    # dataset_noise = Field(str, mandatory=True)
    # class_balance = Field(str, mandatory=True)
    classification_categories = Field(int, default=None)
    
# ______________________________________________________________
class ComputeConstraints(Fact):
    # compute_budget = Field(str, mandatory=True)
    has_gpu = Field(bool, mandatory=True, default=True)
    gpu_memory_gb = Field(float, default=8.0)
    ram_gb = Field(float, default=16.0)
    time_budget_hours = Field(float, mandatory=True)
    search_space_dimensions = Field(int, mandatory=True)
    trial_cost = Field(str, mandatory=True)

class OptimizationBudget(Fact):
    max_trials = Field(int, mandatory=True)
    early_stopping_enabled = Field(bool, default=True)
    max_total_runtime_hours = Field(float, mandatory=True)
# ______________________________________________________________________
class ReasoningStage(Fact):
    current = Field(str, mandatory=True)

class HPOmethod(Fact):
    method = Field(str, mandatory = True)
    score = Field (int, default = 0)
# -----------------------------------------------------------------------
class HardWareSpecs(Fact):
    has_gpu = Field(bool, mandatory=True, default=True)
    

class CurrentTrainingConfig(Fact):
    optimizer = Field(str, mandatory=True)
    learning_rate = Field(float, mandatory=True)
    batch_size = Field(int, mandatory=True)
    weight_decay = Field(float, default=0.0)
    dropout_rate = Field(float, default=0.0)
    scheduler = Field(str, default="none")
    gradient_clipping = Field(bool, default=False)


class GPULookUp(Fact):
    model = Field(str, mandatory=True)
    vram = Field(int, mandatory=True)
    score = Field(int, mandatory=True)

class ComputationPower(Fact):
    total_vram = Field(int, mandatory= True)
    max_vram = Field(int, mandatory = True)
    connected_gpus = Field(bool, mandatory = True)
    

In [6]:
from enum import Enum

class ReasoningStageId(str, Enum):
    INIT = "init"
    HPO_METHOD = "hpo_method"
    SEARCH_SPACE = "search_space"
    RANGES = "ranges"

class HPOMethodCategory(str, Enum):
    EXHAUSTIVE = "exhaustive"  # grid
    SAMPLING = "sampling"  # random
    MODEL_BASED = "model_based"  # bayesian
    MULTIFIDELITY = "multifidelity"  # hyperband
    POPULATION = "population"  # PBT

class HPOMethod(str, Enum):
    GRID_SEARCH = "grid_search"
    RANDOM_SEARCH = "random_search"
    BAYESIAN_OPTIMIZATION = "bayesian_optimization"
    HYPERBAND = "hyperband"
    POPULATION_BASED_TRAINING = "population_based_training"

class ArchitectureType(str, Enum):
    MLP = "mlp"
    CNN = "cnn"
    RNN = "rnn"
    TRANSFORMER = "transformer"
    LLM = "llm"
    


In [7]:
# class ComputationPower(KnowledgeEngine):
#     @DefFacts
#     def load_gpu_look_ups(self):
#         yield 

In [8]:
class HOPEngine(KnowledgeEngine):

    @Rule(
        ComputeConstraints(
            search_space_dimensions=MATCH.d,
            trial_cost=MATCH.cost,
            time_budget_hours=MATCH.hours,
        ),
        NOT(HPOmethod(method = L(lambda method: method == HPOMethod.GRID.SEARCH.value))),
        salience=85,
    )
    def method_grid_search(self, d, cost, hours):
        self.declare(
            HPOMethodChoice(
                method=HPOMethod.GRID_SEARCH.value,
            )
        )

    @Rule(
        ReasoningStage(current=ReasoningStageId.HPO_METHOD.value),
        ComputeConstraints(
            search_space_dimensions=MATCH.d,
            trial_cost=MATCH.cost,
            compute_budget=MATCH.budget,
            time_budget_hours=MATCH.h,
        ),
        salience=84,
    )
    def method_bayesian(self, d, cost, budget, h):
        self.declare(
            HPOMethodChoice(
                method=HPOMethod.BAYESIAN_OPTIMIZATION.value,
            )
        )

In [9]:
engine = HOPEngine()
engine.reset()
engine.run()
engine.facts

FactList([(0, InitialFact())])